# Unified Modelling Pipeline

Replicating the **unified_modelling_pipeline.ipynb** approach directly from `final_clean_dataset_long.csv`.

**Pipeline**: `SimpleImputer(median)` → model (via `sklearn.Pipeline`) · `clone()` per fold

**CV**: 5-fold `TimeSeriesSplit` (pooled) + per-catalogue segment-specific CV

**Evaluation**: OOF predictions → RMSE / MAE / MAPE / R² · Unified vs Segment-specific comparison

### 2.1  Load Data & Build One-Week-Ahead Target

In [ ]:
import warnings, json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from IPython.display import display

warnings.filterwarnings('ignore')
SEED = 42

DATA_PATH_U = Path('../data/processed/final_clean_dataset_long.csv')
OUTPUT_DIR_U = Path('../data/results')

TARGET_BASE = 'price_mid_lkr'
TARGET_U    = 'price_mid_lkr_t_plus_1w'
SEGMENT_COL_U = 'catalogue'
GROUP_U     = ['catalogue', 'grade', 'tier']

df_u = pd.read_csv(DATA_PATH_U)
df_u = df_u.replace([np.inf, -np.inf], np.nan)
df_u = df_u.dropna(subset=[TARGET_BASE, SEGMENT_COL_U]).copy()

# Chronological sort (same as unified_modelling_pipeline.ipynb)
df_u = df_u.sort_values(['sale_year', 'sale_number', 'catalogue', 'grade', 'tier']).reset_index(drop=True)

# One-week-ahead target within each (catalogue, grade, tier) product stream
df_u[TARGET_U] = df_u.groupby(GROUP_U)[TARGET_BASE].shift(-1)
df_u = df_u.dropna(subset=[TARGET_U]).copy().reset_index(drop=True)

print(f'Loaded: {DATA_PATH_U.name}  |  shape after t+1 construction: {df_u.shape}')
print(f'\nSegment distribution (catalogue):')
for seg in sorted(df_u[SEGMENT_COL_U].unique()):
    print(f'  {seg}: {(df_u[SEGMENT_COL_U]==seg).sum():,}')

Loaded: final_clean_dataset_long.csv  |  shape after t+1 construction: (11739, 27)

Segment distribution (catalogue):
  dust: 2,170
  high_grown: 2,120
  low_grown: 5,332
  off_grade: 2,117


### 2.2  Feature Engineering

In [2]:
# ── Rank maps (identical to unified_modelling_pipeline.ipynb) ────────────────
grade_rank_map = {
    'bop': 4, 'bopf': 4, 'bop1': 4,
    'fbop': 3, 'fbop1': 3, 'fbopf': 3, 'fbopf1': 3, 'fbopf_tippy': 3,
    'op': 2, 'op1': 2, 'opa': 2,
    'pekoe': 1, 'pek1': 1, 'pekoe_fbop': 1,
}
tier_rank_map      = {'select_best': 4, 'best': 3, 'below_best': 2, 'others': 1}
elevation_rank_map = {'high_grown': 3, 'medium_grown': 2, 'low_grown': 1,
                       'high': 3, 'medium': 2, 'low': 1}

df_u['grade_rank']    = df_u['grade'].astype(str).str.lower().map(grade_rank_map).fillna(0)
df_u['tier_rank']     = df_u['tier'].astype(str).str.lower().map(tier_rank_map).fillna(0)
df_u['prestige_rank'] = df_u['elevation'].astype(str).str.lower().map(elevation_rank_map).fillna(0)

# Price lag features within each product stream
for _lag in [1, 2, 3]:
    df_u[f'price_lag{_lag}'] = df_u.groupby(GROUP_U)[TARGET_BASE].shift(_lag)

df_u['price_roll_mean3'] = (
    df_u.groupby(GROUP_U)[TARGET_BASE]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)
df_u['price_roll_std3'] = (
    df_u.groupby(GROUP_U)[TARGET_BASE]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=2).std().fillna(0))
)

# Label-encode categoricals
for _col in ['catalogue', 'grade', 'tier', 'region', 'segment',
             'elevation', 'text_crop_change', 'category']:
    if _col in df_u.columns:
        df_u[f'{_col}_enc'] = LabelEncoder().fit_transform(
            df_u[_col].fillna('unknown').astype(str)
        )

# Absolute time index
df_u['time_idx'] = df_u.groupby(['sale_year', 'sale_number']).ngroup()

# ── Feature list (all numeric, exclude price targets + raw strings) ───────────
_drop = {TARGET_BASE, TARGET_U, 'sale_id', 'sale_date_raw',
         'catalogue', 'grade', 'tier', 'region', 'segment',
         'elevation', 'text_crop_change', 'category', 'text_keywords'}
feature_cols_u = [c for c in df_u.select_dtypes(include=[np.number]).columns
                  if c not in _drop]

print(f'Total features for unified model: {len(feature_cols_u)}')
print(feature_cols_u)

Total features for unified model: 31
['temperature_2m_mean_mean', 'precipitation_sum_total_lag1', 'sunshine_duration_total_lag1', 'temperature_2m_mean_mean_lag1', 'precipitation_sum_total_lag2', 'sunshine_duration_total_lag2', 'temperature_2m_mean_mean_lag2', 'precipitation_sum_total_lag3', 'sunshine_duration_total_lag3', 'temperature_2m_mean_mean_lag3', 'sale_number', 'sale_year', 'avg_weather_severity', 'fx_usd', 'grade_rank', 'tier_rank', 'prestige_rank', 'price_lag1', 'price_lag2', 'price_lag3', 'price_roll_mean3', 'price_roll_std3', 'catalogue_enc', 'grade_enc', 'tier_enc', 'region_enc', 'segment_enc', 'elevation_enc', 'text_crop_change_enc', 'category_enc', 'time_idx']


### 2.3  Model Definitions

In [3]:
# ── Models (same hyperparameters as unified_modelling_pipeline.ipynb) ─────────
models_u = {
    'XGBoost': XGBRegressor(
        n_estimators=250, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=-1, verbosity=0,
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=250, learning_rate=0.05, max_depth=-1,
        random_state=SEED, n_jobs=-1, verbose=-1,
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=250, learning_rate=0.05, max_depth=3,
        random_state=SEED,
    ),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, random_state=SEED, n_jobs=-1,
    ),
}

def compute_metrics_u(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-9))) * 100
    r2   = r2_score(y_true, y_pred)
    return rmse, mae, mape, r2

print(f'Models: {list(models_u.keys())}')

Models: ['XGBoost', 'LightGBM', 'Gradient Boosting', 'Random Forest']


### 2.4  5-Fold Unified TimeSeriesSplit CV

In [4]:
# ── Prepare unified matrices ─────────────────────────────────────────────────
X_u = df_u[feature_cols_u].copy()
y_u = pd.to_numeric(df_u[TARGET_U], errors='coerce')
_valid = y_u.notna()
X_u, y_u = X_u[_valid].reset_index(drop=True), y_u[_valid].reset_index(drop=True)
df_sup_u = df_u[_valid].reset_index(drop=True)

print(f'Unified dataset: {len(X_u):,} rows x {len(feature_cols_u)} features')
print(f'Running 5-fold TimeSeriesSplit CV (pooled across all catalogues)...\n')

tscv_u = TimeSeriesSplit(n_splits=5)
all_fold_rows_u = []
oof_preds_u = {n: np.full(len(y_u), np.nan) for n in models_u}

for _fold, (_tr, _te) in enumerate(tscv_u.split(X_u), start=1):
    X_tr, X_te = X_u.iloc[_tr], X_u.iloc[_te]
    y_tr, y_te = y_u.iloc[_tr], y_u.iloc[_te]
    for _name, _model in models_u.items():
        _pipe = Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('model', clone(_model))])
        _pipe.fit(X_tr, y_tr)
        _preds = _pipe.predict(X_te)
        oof_preds_u[_name][_te] = _preds
        _rmse, _mae, _mape, _r2 = compute_metrics_u(y_te, _preds)
        all_fold_rows_u.append({'Fold': _fold, 'Model': _name,
                                'RMSE': _rmse, 'MAE': _mae,
                                'MAPE': _mape, 'R2': _r2,
                                'n_train': len(_tr), 'n_test': len(_te)})

fold_results_u = pd.DataFrame(all_fold_rows_u)

summary_u = (
    fold_results_u.groupby('Model', as_index=False)
    .agg(RMSE_mean=('RMSE','mean'), RMSE_std=('RMSE','std'),
         MAE_mean=('MAE','mean'),   MAE_std=('MAE','std'),
         MAPE_mean=('MAPE','mean'), MAPE_std=('MAPE','std'),
         R2_mean=('R2','mean'),     R2_std=('R2','std'))
    .sort_values('RMSE_mean').reset_index(drop=True)
)
summary_u['Forecast_Horizon'] = 't_plus_1_week'

print('Unified 5-Fold CV Summary (sorted by RMSE):')
display(summary_u.round(4))

Unified dataset: 11,739 rows x 31 features
Running 5-fold TimeSeriesSplit CV (pooled across all catalogues)...

Unified 5-Fold CV Summary (sorted by RMSE):


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,Forecast_Horizon
0,LightGBM,137.9538,19.8319,83.3593,12.0560,6.8941,1.0297,0.9515,0.0087,t_plus_1_week
1,Gradient Boosting,140.1436,19.3521,89.2955,11.0138,7.4500,0.8019,0.9499,0.0093,t_plus_1_week
2,XGBoost,141.9150,21.1160,83.6429,10.6001,6.8190,0.9777,0.9480,0.0137,t_plus_1_week
3,Random Forest,142.2829,21.3988,82.7038,11.9608,6.7220,0.9529,0.9483,0.0108,t_plus_1_week


### 2.5  All-Model Findings Table

In [5]:
# ── All-model findings table (RMSE rank + R² rank + CV stability) ────────────
# Mirrors the 'unified_all_model_findings' table from unified_modelling_pipeline.ipynb
findings_u = summary_u.copy()
findings_u['RMSE_rank']   = findings_u['RMSE_mean'].rank(method='dense', ascending=True).astype(int)
findings_u['R2_rank']     = findings_u['R2_mean'].rank(method='dense', ascending=False).astype(int)
findings_u['RMSE_cv_pct'] = (findings_u['RMSE_std'] / findings_u['RMSE_mean']) * 100
findings_u['R2_cv_pct']   = (
    findings_u['R2_std'].abs()
    / findings_u['R2_mean'].abs().replace(0, np.nan)
) * 100
findings_u = findings_u.sort_values(['RMSE_rank', 'R2_rank']).reset_index(drop=True)

print('All-Model Findings Table (ranks + CV stability):')
display(findings_u.round(4))

best_model_name_u = summary_u.iloc[0]['Model']
best_row_u = summary_u.iloc[0]
print(f'\nBest unified model: {best_model_name_u}  '
      f'(RMSE={best_row_u["RMSE_mean"]:.2f} ± {best_row_u["RMSE_std"]:.2f}  '
      f'R²={best_row_u["R2_mean"]:.4f})')

All-Model Findings Table (ranks + CV stability):


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,Forecast_Horizon,RMSE_rank,R2_rank,RMSE_cv_pct,R2_cv_pct
0,LightGBM,137.9538,19.8319,83.3593,12.0560,6.8941,1.0297,0.9515,0.0087,t_plus_1_week,1,1,14.3757,0.9135
1,Gradient Boosting,140.1436,19.3521,89.2955,11.0138,7.4500,0.8019,0.9499,0.0093,t_plus_1_week,2,2,13.8088,0.9744
2,XGBoost,141.9150,21.1160,83.6429,10.6001,6.8190,0.9777,0.9480,0.0137,t_plus_1_week,3,4,14.8793,1.4428
3,Random Forest,142.2829,21.3988,82.7038,11.9608,6.7220,0.9529,0.9483,0.0108,t_plus_1_week,4,3,15.0396,1.1371



Best unified model: LightGBM  (RMSE=137.95 ± 19.83  R²=0.9515)


### 2.6  Segment-Specific CV (per catalogue)

In [6]:
# ── Segment-specific 5-fold CV for each catalogue ────────────────────────────
# Mirrors the High-Grown / Low-Grown specific CV in unified_modelling_pipeline.ipynb
# but extended to all 4 catalogues (high_grown, low_grown, off_grade, dust).
print('Running segment-specific TimeSeriesSplit CV per catalogue...\n')

seg_summaries_u   = {}   # catalogue -> CV summary DataFrame
best_seg_models_u = {}   # catalogue -> best model name
oof_preds_seg_u   = {}   # catalogue -> {model_name -> oof array}
seg_train_data_u  = {}   # catalogue -> {X, y, df}

for _cat in sorted(df_sup_u[SEGMENT_COL_U].unique()):
    _seg_mask = df_sup_u[SEGMENT_COL_U] == _cat
    _X_s  = X_u[_seg_mask.values].reset_index(drop=True)
    _y_s  = y_u[_seg_mask.values].reset_index(drop=True)
    _df_s = df_sup_u[_seg_mask].reset_index(drop=True)

    if len(_X_s) < 50:
        print(f'Skipping {_cat}: only {len(_X_s)} rows.')
        continue

    _tscv_s = TimeSeriesSplit(n_splits=5)
    _seg_fold_rows = []
    oof_preds_seg_u[_cat] = {n: np.full(len(_y_s), np.nan) for n in models_u}

    for _fold, (_tr, _te) in enumerate(_tscv_s.split(_X_s), start=1):
        for _name, _model in models_u.items():
            _pipe = Pipeline([('impute', SimpleImputer(strategy='median')),
                              ('model', clone(_model))])
            _pipe.fit(_X_s.iloc[_tr], _y_s.iloc[_tr])
            _preds = _pipe.predict(_X_s.iloc[_te])
            oof_preds_seg_u[_cat][_name][_te] = _preds
            _rm, _ma, _mp, _r2 = compute_metrics_u(_y_s.iloc[_te], _preds)
            _seg_fold_rows.append({'Fold': _fold, 'Model': _name,
                                   'RMSE': _rm, 'MAE': _ma,
                                   'MAPE': _mp, 'R2': _r2})

    _seg_sum = (
        pd.DataFrame(_seg_fold_rows)
        .groupby('Model', as_index=False)
        .agg(RMSE_mean=('RMSE','mean'), RMSE_std=('RMSE','std'),
             MAE_mean=('MAE','mean'),   MAE_std=('MAE','std'),
             MAPE_mean=('MAPE','mean'), MAPE_std=('MAPE','std'),
             R2_mean=('R2','mean'),     R2_std=('R2','std'))
        .sort_values('RMSE_mean').reset_index(drop=True)
    )

    # Combined rank selection (same logic as unified_modelling_pipeline.ipynb)
    _sel = _seg_sum.copy()
    _sel['RMSE_rank'] = _sel['RMSE_mean'].rank(method='dense', ascending=True).astype(int)
    _sel['R2_rank']   = _sel['R2_mean'].rank(method='dense', ascending=False).astype(int)
    _sel['combined']  = _sel['RMSE_rank'] + _sel['R2_rank']
    _best = _sel.sort_values(['combined', 'RMSE_std', 'RMSE_mean']).iloc[0]['Model']

    seg_summaries_u[_cat]   = _seg_sum
    best_seg_models_u[_cat] = _best
    seg_train_data_u[_cat]  = {'X': _X_s, 'y': _y_s, 'df': _df_s}

    print(f'{_cat.upper().replace("_"," ")} (n={len(_X_s):,}):')
    display(_seg_sum.round(3))
    print(f'  >> Best: {_best}\n')

Running segment-specific TimeSeriesSplit CV per catalogue...

DUST (n=2,170):


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std
0,Random Forest,108.350,46.956,75.364,29.273,7.368,2.397,0.624,0.342
1,XGBoost,110.955,47.850,77.308,29.002,7.513,2.317,0.606,0.366
2,LightGBM,112.271,39.481,78.375,25.537,7.674,2.170,0.613,0.291
3,Gradient Boosting,116.699,34.982,85.767,25.104,8.442,2.191,0.593,0.243


  >> Best: Random Forest

HIGH GROWN (n=2,120):


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std
0,LightGBM,159.974,48.384,102.721,20.178,8.601,1.674,0.461,0.218
1,XGBoost,160.217,47.901,98.338,18.473,8.201,1.651,0.455,0.224
2,Gradient Boosting,163.434,51.373,101.559,17.922,8.449,1.285,0.430,0.263
3,Random Forest,170.220,55.328,103.378,28.678,8.582,2.465,0.354,0.402


  >> Best: LightGBM

LOW GROWN (n=5,332):


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std
0,LightGBM,140.434,34.295,74.486,17.577,4.101,0.857,0.967,0.009
1,Gradient Boosting,142.986,39.679,74.933,20.962,4.076,0.884,0.966,0.011
2,XGBoost,145.692,35.091,75.949,18.131,4.098,0.847,0.965,0.009
3,Random Forest,147.335,39.769,77.391,19.129,4.230,0.794,0.964,0.012


  >> Best: LightGBM

OFF GRADE (n=2,117):


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std
0,LightGBM,101.354,17.566,77.820,14.024,9.593,2.298,0.484,0.167
1,XGBoost,101.934,17.235,80.401,14.959,9.971,2.625,0.483,0.135
2,Random Forest,103.422,18.323,81.101,15.362,10.103,2.752,0.468,0.144
3,Gradient Boosting,106.328,18.064,82.449,15.824,10.229,2.825,0.441,0.129


  >> Best: LightGBM



In [ ]:
# ── Save results (mirrors Section 8 of unified_modelling_pipeline.ipynb) ─────
fold_results_u.to_csv(OUTPUT_DIR_U / 'unified_cv_fold_results.csv', index=False)
summary_u.to_csv(OUTPUT_DIR_U / 'unified_cv_summary.csv', index=False)
findings_u.to_csv(OUTPUT_DIR_U / 'unified_all_model_findings.csv', index=False)

for _cat, _df_s in seg_summaries_u.items():
    _df_s.to_csv(OUTPUT_DIR_U / f'segment_cv_summary_{_cat}.csv', index=False)

print('Saved to data/processed-2024/:')
print('  unified_cv_fold_results.csv')
print('  unified_cv_summary.csv')
print('  unified_all_model_findings.csv')
print('  unified_vs_segment_comparison.csv')
for _cat in seg_summaries_u:
    print(f'  segment_cv_summary_{_cat}.csv')

Saved to data/processed-2024/:
  unified_cv_fold_results.csv
  unified_cv_summary.csv
  unified_all_model_findings.csv
  unified_vs_segment_comparison.csv
  segment_cv_summary_dust.csv
  segment_cv_summary_high_grown.csv
  segment_cv_summary_low_grown.csv
  segment_cv_summary_off_grade.csv
